# Depletion capacitance and C–V profiling

The notebook derives capacitance from the depletion width and checks numerical doping extraction.

In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "figures" / "src").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Could not locate lecture-notes/figures/src")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "figures" / "src"))
from scipy.constants import elementary_charge as q, Boltzmann as k_B, epsilon_0

NA, ND, ni, T = 1e24, 2e21, 1e16, 300.0
eps = 11.7*epsilon_0
VR = np.linspace(0,12,301)
Vbi = k_B*T/q*np.log(NA*ND/ni**2)
W = np.sqrt(2*eps*(Vbi+VR)/q*(1/NA+1/ND))
C = eps/W
slope, intercept = np.polyfit(VR,1/C**2,1)
N_eff = 2/(q*eps*slope)
print(f"linear-fit effective doping = {N_eff/1e6:.4e} cm^-3")

linear-fit effective doping = 1.9960e+15 cm^-3


In [2]:
from semphy.junctions import depletion_capacitance, extract_one_sided_doping

C_api = depletion_capacitance(VR,NA,ND,ni,T,eps)
N_cv = extract_one_sided_doping(VR,C_api,eps)
expected = NA*ND/(NA+ND)
print(f"maximum C–V extraction error = {np.max(abs(N_cv-expected))/expected:.3e}")
assert np.all(np.diff(C_api)<0)
assert np.max(abs(N_cv-expected))/expected < 1e-10

maximum C–V extraction error = 1.605e-13


In [3]:
# Refinement verifies that centered differentiation preserves the exact linear relation.
for count in (31,101,301):
    v=np.linspace(0,12,count)
    c=depletion_capacitance(v,NA,ND,ni,T,eps)
    n=extract_one_sided_doping(v,c,eps)
    print(count, np.max(abs(n-expected))/expected)

31 2.101346304e-14
101 2.62668288e-14
301 1.6049032396800001e-13


In [4]:
from semphy.junctions import capacitance_figure
from semphy.junction_atlas import capacitance_construction_atlas

capacitance_figure()
capacitance_construction_atlas()
print("wrote C–V validation and original capacitance-construction plate")

wrote C–V validation and original capacitance-construction plate
